# Gradient Surgery with Tangent

Tangent differentiates by **source-to-source transformation**: `tangent.grad(f)`
returns an ordinary Python function whose source you can *read*, and whose
backward pass you can *edit*. Tracing autodiff (JAX, PyTorch, TensorFlow) builds
an opaque graph or tape at run time — there is no readable adjoint to open up.

This notebook shows the thing no tracing AD can copy: **read the generated
gradient, then operate on it** — scale it, clip it, log it, guard it — with
`tangent.insert_grad_of`, a first-class context manager that splices code into
the backward pass and *survives the optimizer*.


In [1]:
import numpy as np
import tangent


## 1. Read the generated gradient

Every gradient carries its own source in `__tangent_source__`.

In [2]:
def f(x):
    y = x * x
    return y * 3.0

df = tangent.grad(f)
print("f'(2.0) =", df(2.0))          # 12.0
print()
print(df.__tangent_source__)


f'(2.0) = 12.0

def dfdx(x, by_times_3_0=1.0):
    y = x * x
    y_times_3_0 = y * 3.0
    # Reconcile the seed with the structure of the return value
    by_times_3_0 = tangent.match_seed(y_times_3_0, by_times_3_0)
    assert tangent.shapes_match(y_times_3_0, by_times_3_0), 'Shape mismatch between return value (%s) and seed derivative (%s)' % (tangent.get_shape(y_times_3_0), tangent.get_shape(by_times_3_0))
    # Grad of: y = x * x
    _by = tangent.unbroadcast(by_times_3_0 * 3.0, y)
    by = _by
    _bx = tangent.unbroadcast(by * x, x)
    _bx2 = tangent.unbroadcast(by * x, x)
    bx = _bx
    bx = tangent.add_grad(bx, _bx2)
    return bx



## 2. `explain()` — the whole debuggability story in one call

`tangent.explain` prints the primal, the generated adjoint, a finite-difference
check, the primal's **data-flow graph**, and flags any argument that provably
never affects the output (a *static* claim for straight-line code).

In [3]:
def model(x, unused_bias):
    a = x * x
    b = a * 3.0
    return np.sum(b)          # unused_bias never reaches the output

_ = tangent.explain(model, np.array([1.0, 2.0]), 5.0, wrt=(0, 1))


────────────────────────────────────────────────────────────────────────
PRIMAL  model(x, unused_bias)
────────────────────────────────────────────────────────────────────────
def model(x, unused_bias):
    a = x * x
    b = a * 3.0
    return np.sum(b)          # unused_bias never reaches the output
────────────────────────────────────────────────────────────────────────
GRADIENT (generated source)
────────────────────────────────────────────────────────────────────────
def dmodeldxunused_bias(x, unused_bias, bnp_sum_b=1.0):
    a = x * x
    b = a * 3.0
    np_sum_b = np.sum(b)
    bunused_bias = tangent.init_grad(unused_bias)
    # Reconcile the seed with the structure of the return value
    bnp_sum_b = tangent.match_seed(np_sum_b, bnp_sum_b)
    assert tangent.shapes_match(np_sum_b, bnp_sum_b), 'Shape mismatch between return value (%s) and seed derivative (%s)' % (tangent.get_shape(np_sum_b), tangent.get_shape(bnp_sum_b))
    # Grad of: b = a * 3.0
    _bb = tangent.astype(tangent

Note the `[dead: never affects output]` edge and the static note for
argument 1 — Tangent read this off the data-flow graph, not from a lucky
zero at one point.

## 3. Gradient surgery: scale the gradient

`with tangent.insert_grad_of(y) as dy:` binds `dy` to the gradient flowing into
`y` **during the backward pass**. Assign to it and you have edited the
gradient. Here we scale it by 0.9, so `f'(2)` drops from 12 to 10.8.

In [4]:
def f_scaled(x):
    y = x * x
    with tangent.insert_grad_of(y) as dy:
        dy = dy * 0.9          # <-- surgery: scale the gradient into y
    return y * 3.0

print("scaled f'(2.0) =", tangent.grad(f_scaled)(2.0), "  (12 * 0.9 = 10.8)")
print()
print(tangent.grad(f_scaled).__tangent_source__)


scaled f'(2.0) = 10.8   (12 * 0.9 = 10.8)

def df_scaleddx(x, by_times_3_0=1.0):
    y = x * x
    y_times_3_0 = y * 3.0
    # Reconcile the seed with the structure of the return value
    by_times_3_0 = tangent.match_seed(y_times_3_0, by_times_3_0)
    assert tangent.shapes_match(y_times_3_0, by_times_3_0), 'Shape mismatch between return value (%s) and seed derivative (%s)' % (tangent.get_shape(y_times_3_0), tangent.get_shape(by_times_3_0))
    # Grad of: dy = dy * 0.9
    _by = tangent.unbroadcast(by_times_3_0 * 3.0, y)
    by = _by
    # Inserted code
    by = by * 0.9
    # Grad of: y = x * x
    _bx = tangent.unbroadcast(by * x, x)
    _bx2 = tangent.unbroadcast(by * x, x)
    bx = _bx
    bx = tangent.add_grad(bx, _bx2)
    return bx



The `* 0.9` appears verbatim in the generated backward pass, tagged
`# Inserted code`.

## 4. The surgery survives optimization

DCE and the other passes never strip injected gradient code.

In [5]:
opt = tangent.grad(f_scaled).__tangent_source__
noopt = tangent.grad(f_scaled, optimized=False).__tangent_source__
print("injected '* 0.9' present with    optimization:", '0.9' in opt)
print("injected '* 0.9' present without optimization:", '0.9' in noopt)


injected '* 0.9' present with    optimization: True
injected '* 0.9' present without optimization: True


## 5. Gradient clipping — a real use

Clip the (large) gradient flowing into an intermediate. Here the unclipped
gradient is 192; clipping the gradient into `y` to `[-5, 5]` brings it to 20.

In [6]:
def loss(x):
    y = x * x
    return np.sum(y * y * y)            # d/dy = 3 y^2 -> large

def loss_clipped(x):
    y = x * x
    with tangent.insert_grad_of(y) as dy:
        dy = np.clip(dy, -5.0, 5.0)     # <-- clip the gradient into y
    return np.sum(y * y * y)

x = np.array([2.0])
print("unclipped grad:", tangent.grad(loss)(x))
print("clipped grad:  ", tangent.grad(loss_clipped)(x))


unclipped grad: [192.]
clipped grad:   [20.]


## 6. Log / inspect a gradient mid-backward

Because the injected code is ordinary Python, you can simply `print` (or set a
breakpoint, or assert) inside the backward pass.

In [7]:
def f_logged(x):
    y = np.sin(x)
    with tangent.insert_grad_of(y) as dy:
        print("  [backward] gradient flowing into y =", dy)
    return np.sum(y * y)

print("calling the gradient function:")
g = tangent.grad(f_logged)(np.array([0.0, 1.0]))
print("result:", g)


calling the gradient function:
  [backward] gradient flowing into y = [0.         1.68294197]
result: [0.         0.90929743]


## 7. `source_map`: adjoint lines ↔ primal lines

Every generated line is traced back to the primal statement it differentiates,
so a message about the backward pass points at *your* code.

In [8]:
df = tangent.grad(f)
for entry in tangent.source_map(df, f):
    if entry['primal']:
        print("gen L%-2d  <-  primal: %s" % (entry['line'], entry['primal']))


gen L7   <-  primal: y = x * x
gen L8   <-  primal: y = x * x
gen L9   <-  primal: y = x * x
gen L10  <-  primal: y = x * x
gen L11  <-  primal: y = x * x
gen L12  <-  primal: y = x * x
gen L13  <-  primal: y = x * x
gen L14  <-  primal: y = x * x


## Why this is unique

Reading and editing a readable adjoint — scaling, clipping, logging, guarding —
is only possible because Tangent emits real Python source. A tracing autodiff
has no adjoint source to open, so this workflow has no equivalent in JAX,
PyTorch, or TensorFlow. `insert_grad_of` makes it a supported, optimizer-stable
API rather than a hack.
